In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

np.random.seed(189)

In [2]:
# Read the dataset
data = scipy.io.loadmat('data.mat')
X = data['X']
y = data['y'].flatten()
X_test = data['X_test']

# Split the data into training and validation sets
# Here we need to split the train-val set before normalization
# otherwise, information from the validation set would leak into the training set
train_size = 0.8
num_train = int(train_size * X.shape[0])
indices = np.random.permutation(X.shape[0])
train_indices = indices[:num_train]
val_indices = indices[num_train:]
X_train = X[train_indices]
y_train = y[train_indices]
X_val = X[val_indices]
y_val = y[val_indices]

# normalize the data
# Use training statistics to normalize both training and val/test data.
X_train_mean = np.mean(X_train, axis=0)
X_train_std = np.std(X_train, axis=0)
X_train = (X_train - X_train_mean) / X_train_std
X_val = (X_val - X_train_mean) / X_train_std
X_test = (X_test - X_train_mean) / X_train_std

# Add fictitious feature
X_train = np.hstack((X_train, np.ones((X_train.shape[0], 1))))
X_val = np.hstack((X_val, np.ones((X_val.shape[0], 1))))
X_test = np.hstack((X_test, np.ones((X_test.shape[0], 1))))
print(f"Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}")
print(f"Validation data shape: {X_val.shape}, Validation labels shape: {y_val.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (4000, 13), Training labels shape: (4000,)
Validation data shape: (1000, 13), Validation labels shape: (1000,)
Test data shape: (1000, 13)


In [3]:
def cost_func(X, y, w, lambda_):
    '''
    J(w) = - y^T * log(s) - (1 - y)^T * log(1 - s)] + lambda * ||w||^2
    '''
    sigmoid = scipy.special.expit(X @ w)
    log_likelihood = - (y.T @ np.log(sigmoid)) \
                     - ((1 - y).T @ np.log(1 - sigmoid))
    regularization = lambda_ * np.linalg.norm(w) ** 2
    return log_likelihood + regularization

In [4]:
# Batch Gradient Descent
def batch_gradient_descent(X, y, learning_rate, lambda_, max_iters):
    '''
    w <- (1 - 2 * lambda * learning_rate) * w + learning_rate * X^T * (y - s)
    '''
    num_samples, num_features = X.shape
    w = np.ones(num_features)  # Initialize weights
    for iter in range(max_iters):
        sigmoid = scipy.special.expit(X @ w)
        w = (1 - 2 * lambda_ * learning_rate) * w \
            + learning_rate * (X.T @ (y - sigmoid))
    return w

In [5]:
# Tune the hyperparameters
learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]
lambdas = [0.0001, 0.001, 0.01, 0.1, 1]
max_iters = 10000

best_val_accuracy = 0
best_hyperparams = (None, None)
for learning_rate in learning_rates:
    for lambda_ in lambdas:
        w = batch_gradient_descent(X_train, y_train, learning_rate, lambda_, max_iters)
        val_predictions = scipy.special.expit(X_val @ w) >= 0.5
        val_accuracy = np.mean(val_predictions == y_val)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_hyperparams = (learning_rate, lambda_)
print("Best Val Accuracy: {:.4f} with Learning Rate = {} and Lambda = {}"
            .format(best_val_accuracy, best_hyperparams[0], best_hyperparams[1]))

Best Val Accuracy: 0.9980 with Learning Rate = 1e-05 and Lambda = 0.0001


In [ ]:
# Best hyperparameters found for batch gradient descent
learning_rate_bgd = 1e-5
lambda_bgd = 0.0001

max_iters_bgd = 10000
iterations_bgd = np.arange(max_iters_bgd)
w = np.ones(X_train.shape[1]) # Initialize weights
train_costs_bgd = []
val_costs_bgd = []
for iter in range(max_iters_bgd):
    w = (1 - 2 * lambda_bgd * learning_rate_bgd) * w \
        + learning_rate_bgd * (X_train.T @ (y_train - scipy.special.expit(X_train @ w)))
    train_cost = cost_func(X_train, y_train, w, lambda_bgd)
    val_cost = cost_func(X_val, y_val, w, lambda_bgd)
    train_costs_bgd.append(train_cost)
    val_costs_bgd.append(val_cost)

# Plot the training and validation cost over iterations
plt.plot(iterations_bgd, train_costs_bgd, label='Training Cost')
plt.plot(iterations_bgd, val_costs_bgd, label='Validation Cost')
plt.xlabel('number of iterations')
plt.ylabel('Cost function')
plt.title('Batch Gradient Descent')
plt.legend()
plt.savefig('q3_wine_bgd_plot.png')
plt.show()

In [7]:
# Stochastic Gradient Descent
def stochastic_gradient_descent(X, y, learning_rate, lambda_, max_iters):
    '''
    w <- (1 - 2 * lambda * learning_rate) * w + learning_rate * (y_i - s_i) * x_i 
    '''
    num_samples, num_features = X.shape
    w = np.ones(num_features) # Initialize weights
    for iter in range(max_iters):
        random_index = np.random.randint(num_samples)
        x_i = X[random_index]
        y_i = y[random_index]
        s_i = scipy.special.expit(x_i @ w)
        w = (1 - 2 * lambda_ * learning_rate) * w \
            + learning_rate * (y_i - s_i) * x_i
    return w

In [8]:
# Tune the hyperparameters
learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]
lambdas = [0.0001, 0.001, 0.01, 0.1, 1]
max_iters = 10000

best_val_accuracy = 0
best_hyperparams = (None, None)
for learning_rate in learning_rates:
    for lambda_ in lambdas:
        w = stochastic_gradient_descent(X_train, y_train, learning_rate, lambda_, max_iters)
        val_predictions = scipy.special.expit(X_val @ w) >= 0.5
        val_accuracy = np.mean(val_predictions == y_val)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_hyperparams = (learning_rate, lambda_)
print("Best Val Accuracy: {:.4f} with Learning Rate = {} and Lambda = {}"
            .format(best_val_accuracy, best_hyperparams[0], best_hyperparams[1]))

Best Val Accuracy: 0.9970 with Learning Rate = 0.1 and Lambda = 0.0001


In [ ]:
# Best hyperparameters found for stochastic gradient descent
learning_rate_sgd = 0.1
lambda_sgd = 0.0001

max_iters_sgd = 10000
iterations_sgd = np.arange(max_iters_sgd)
w = np.ones(X_train.shape[1]) # Initialize weights
train_costs_sgd = []
val_costs_sgd = []
for iter in range(max_iters_sgd):
    num_samples = X_train.shape[0]
    random_index = np.random.randint(num_samples)
    x_i = X_train[random_index]
    y_i = y_train[random_index]
    s_i = scipy.special.expit(x_i @ w)
    w = (1 - 2 * lambda_sgd * learning_rate_sgd) * w \
        + learning_rate_sgd * (y_i - s_i) * x_i.T
    train_cost = cost_func(X_train, y_train, w, lambda_sgd)
    val_cost = cost_func(X_val, y_val, w, lambda_sgd)
    train_costs_sgd.append(train_cost)
    val_costs_sgd.append(val_cost)

# Plot the training and validation cost over iterations
plt.plot(iterations_sgd, train_costs_sgd, label='Training Cost')
plt.plot(iterations_sgd, val_costs_sgd, label='Validation Cost')
plt.xlabel('number of iterations')
plt.ylabel('Cost function')
plt.title('Stochastic Gradient Descent')
plt.legend()
plt.savefig('q3_wine_sgd_plot.png')
plt.show()

In [ ]:
# Compare batch gradient descent and stochastic gradient descent
iterations = np.arange(10000)
plt.plot(iterations, train_costs_bgd, label='BGD Training Cost')
plt.plot(iterations, train_costs_sgd, label='SGD Training Cost')
plt.xlabel('number of iterations')
plt.ylabel('Cost function')
plt.title('BGD vs SGD Training Cost Comparison')
plt.legend()
plt.savefig('q3_wine_bgd_vs_sgd_plot.png')
plt.show()

# Here one can find that SGD converges faster than BGD in terms of number of iterations.

In [11]:
# Use a step size that slowly shrinks from iteration to iteration for stochastic gradient descent
# learning_rate_t = delta / t, where delta is a constant hyperparameter and t is the iteration number

def stochastic_gradient_descent_shrink(X, y, delta, lambda_, max_iters):
    '''
    w <- (1 - 2 * lambda * learning_rate_t) * w + learning_rate_t * (y_i - s_i) * x_i 
    where learning_rate_t = delta / t
    '''
    num_samples, num_features = X.shape
    w = np.zeros(num_features) # Initialize weights
    for iter in range(1, max_iters + 1): # start from 1 to avoid division by zero
        learning_rate_t = delta / iter
        random_index = np.random.randint(num_samples)
        x_i = X[random_index]
        y_i = y[random_index]
        s_i = scipy.special.expit(x_i @ w)
        w = (1 - 2 * lambda_ * learning_rate_t) * w \
            + learning_rate_t * (y_i - s_i) * x_i
    return w

In [12]:
# Tune the hyperparameters
deltas = [0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100]
lambdas = [0.0001, 0.001, 0.01, 0.1, 1]
max_iters = 10000

best_val_accuracy = 0
best_hyperparams = (None, None)
for delta in deltas:
    for lambda_ in lambdas:
        w = stochastic_gradient_descent_shrink(X_train, y_train, delta, lambda_, max_iters)
        val_predictions = scipy.special.expit(X_val @ w) >= 0.5
        val_accuracy = np.mean(val_predictions == y_val)
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_hyperparams = (delta, lambda_)
print("Best Val Accuracy: {:.4f} with delta = {} and Lambda = {}"
            .format(best_val_accuracy, best_hyperparams[0], best_hyperparams[1]))

Best Val Accuracy: 0.9960 with delta = 100 and Lambda = 0.001


In [ ]:
# Best hyperparameters found for stochastic gradient descent with shrinking step size
delta_sgd_shrink = 100
lambda_sgd_shrink = 0.01

max_iters_sgd_shrink = 10000
iterations_sgd_shrink = np.arange(max_iters_sgd_shrink)
w = np.ones(X_train.shape[1]) # Initialize weights
train_costs_sgd_shrink = []
val_costs_sgd_shrink = []
for iter in range(1, max_iters_sgd_shrink + 1):
    learning_rate_t = delta_sgd_shrink / iter
    num_samples = X_train.shape[0]
    random_index = np.random.randint(num_samples)
    x_i = X_train[random_index]
    y_i = y_train[random_index]
    s_i = scipy.special.expit(x_i @ w)
    w = (1 - 2 * lambda_sgd_shrink * learning_rate_t) * w \
        + learning_rate_t * (y_i - s_i) * x_i.T
    train_cost = cost_func(X_train, y_train, w, lambda_sgd_shrink)
    val_cost = cost_func(X_val, y_val, w, lambda_sgd_shrink)
    train_costs_sgd_shrink.append(train_cost)
    val_costs_sgd_shrink.append(val_cost)

# Plot the training and validation cost over iterations
plt.plot(iterations_sgd_shrink, train_costs_sgd_shrink, label='Training Cost')
plt.plot(iterations_sgd_shrink, val_costs_sgd_shrink, label='Validation Cost')
plt.xlabel('number of iterations')
plt.ylabel('Cost function')
plt.title('Stochastic Gradient Descent with Shrinking Step Size')
plt.legend()
plt.savefig('q3_wine_sgd_shrink_plot.png')
plt.show()

In [ ]:
# Compare stochastic gradient descent and stochastic gradient descent with shrinking step size
iterations = np.arange(10000)
plt.plot(iterations, train_costs_sgd, label='SGD Training Cost')
plt.plot(iterations, train_costs_sgd_shrink, label='SGD_shrink Training Cost')
plt.xlabel('number of iterations')
plt.ylabel('Cost function')
plt.title('SGD vs SGD_shrink Training Cost Comparison')
plt.legend()
plt.savefig('q3_wine_sgd_vs_sgdshrink_plot.png')
plt.show()

In [15]:
# Kaggle.
# Here I choose SGD as my classifier
# Private Score: 0.994
# Public Score: 0.996
w = stochastic_gradient_descent(X_train, y_train, 
                                learning_rate=0.1, lambda_=0.0001, max_iters=10000)
y_test_pred = scipy.special.expit(X_test @ w) >= 0.5

# save the results into a kaggle accepted csv
import pandas as pd
def results_to_csv(y_test):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1 # Ensures that the index starts at 1
    df.to_csv('submission_wine.csv', index_label='Id')

results_to_csv(y_test_pred)